<a href="https://colab.research.google.com/github/MariyamKassenova/dsa-homework/blob/main/hw05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

In [ ]:
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [ ]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


In [ ]:
## [База] 1. Паспорт таблицы
#Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
#Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

#Сколько строк и столбцов?
rows, cols = df.shape
print(f"Строк: {rows} \nCтолбцов: {cols}\n")
#Какие типы у столбцов?
print(df.dtypes)

#Сколько пропусков в `delivery_days`?
missing_delivery = df["delivery_days"].isna().sum()
print(f"\nПропусков в delivery_days: {missing_delivery}\n")

#Сколько разных значений в `city`, `category`, `status`, `customer_id`?
cols_to_check = ["city", "category", "status", "customer_id"]
print("Количество уникальных значений:")
print(df[cols_to_check].nunique())
print("\nВозможны аномалии в отчетах тк:\n1.Большое кол-во пропусков в delivery_days \n2.Отсутствует унифицированные наименования по городам \n3.Некорректные типы данных в столбцах ")

Строк: 5463 
Cтолбцов: 8

order_id           int64
order_date        object
customer_id       object
city              object
category          object
amount           float64
delivery_days    float64
status            object
dtype: object

Пропусков в delivery_days: 160

Количество уникальных значений:
city             14
category          5
status            2
customer_id    2003
dtype: int64

Возможны аномалии в отчетах тк:
1.Большое кол-во пропусков в delivery_days 
2.Отсутствует унифицированные наименования по городам 
3.Некорректные типы данных в столбцах 


In [ ]:
## [База] 2. Типичный заказ
##Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
##Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

# оплаченные заказы и исключаем тестовые
mask_paid = df["status"] == "оплачен"
mask_no_test = df["customer_id"] != "TEST"
clean_paid = df[mask_paid & mask_no_test]

# среднее, медиана, 25-й и 75-й перцентили суммы.
mean_value = clean_paid["amount"].mean()
median_value = clean_paid["amount"].median()
p25 = clean_paid["amount"].quantile(0.25)
p75 = clean_paid["amount"].quantile(0.75)
# Вывод
print(f"Среднее: {mean_value:.2f}")
print(f"Медиана: {median_value:.2f}")
print(f"25-й перцентиль: {p25:.2f}")
print(f"75-й перцентиль: {p75:.2f}")
print("Типичный чек выглядит по медиане тк лежит между 25м и 75 персентилем")

Среднее: 14672.39
Медиана: 10300.00
25-й перцентиль: 5880.00
75-й перцентиль: 18690.00
Типичный чек выглядит по медиане тк лежит между 25м и 75 персентилем


In [ ]:
## [База] 3. Отмены по городам
##Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
##Какой город выделяется и во сколько раз?

In [ ]:
print(df["city"].unique())
print(df["status"].unique())

['Алматы' 'Астана' 'Караганда' 'Шымкент' 'алматы' 'Атырау' 'шымкент'
 'Shymkent' 'астана' 'Актобе' 'Astana ' 'АЛМАТЫ' ' Алматы' 'Almaty']
['оплачен' 'отменён']


In [ ]:
## [База] 3. Отмены по городам
#Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
#Какой город выделяется и во сколько раз?

# отменненые заказы
is_cancelled = df["status"].isin(["отменён"])
# Доля отмен по всем заказам
total_cancel_rate = is_cancelled.mean()
print(f"Общая доля отмен: {total_cancel_rate:.2%}")

# Доля отмен в Шымкенте
shymkent = ["Шымкент", "шымкент", "Shymkent"]
mask_shymkent = df["city"].str.strip().isin(shymkent)
shymkent_cancel_rate = is_cancelled[mask_shymkent].mean()
print(f"Доля отмен в Шымкенте: {shymkent_cancel_rate:.2%}")
# Доля отмен в Алматы
almaty = ["Алматы", "алматы", "Almaty", "АЛМАТЫ", " Алматы"]
mask_almaty = df["city"].str.strip().isin(almaty)
almaty_cancel_rate = is_cancelled[mask_almaty].mean()
print(f"Доля отмен в Алматы: {almaty_cancel_rate:.2%}")

# доля отмен
ratio_to_total = shymkent_cancel_rate / total_cancel_rate
ratio_to_almaty = shymkent_cancel_rate / almaty_cancel_rate
print(f"Шымкент / Общая: в {ratio_to_total:.2f} раз(а)")
print(f"Шымкент / Алматы: в {ratio_to_almaty:.2f} раз(а)")


Общая доля отмен: 10.23%
Доля отмен в Шымкенте: 20.40%
Доля отмен в Алматы: 7.86%
Шымкент / Общая: в 1.99 раз(а)
Шымкент / Алматы: в 2.59 раз(а)


In [ ]:
CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

In [ ]:
df["city_clean"] = df["city"].str.strip().str.lower().map(CITY)

In [ ]:
print(df["city_clean"].unique())

['Алматы' 'Астана' 'Караганда' 'Шымкент' 'Атырау' 'Актобе']


In [ ]:
## [Стандарт] 4. Октябрь против сентября по двум городам
#Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
#Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

# Оплаченные заказы и исключаем тестовых клиентов
mask_clean = (df["status"] == "оплачен") & (df["customer_id"] != "TEST")
df_clean = df[mask_clean].copy()

# Фильтр по месяцам
mask_sep = df_clean["order_date"] < "2026-10"
mask_oct = df_clean["order_date"] >= "2026-10"

# Выручка в Алматы
rev_sep_almaty = df_clean[mask_sep & (df_clean["city_clean"] == "Алматы")]["amount"].sum()
rev_oct_almaty = df_clean[mask_oct & (df_clean["city_clean"] == "Алматы")]["amount"].sum()
upd_almaty = (rev_oct_almaty - rev_sep_almaty) / rev_sep_almaty
print("Алматы:")
print(f"  Выручка за сентябрь: {rev_sep_almaty:,.2f}")
print(f"  Выручка за октябрь: {rev_oct_almaty:,.2f}")
print(f"  Изменение: {upd_almaty:.2%}\n")

# Выручка в Шымкенте
rev_sep_shymkent = df_clean[mask_sep & (df_clean["city_clean"] == "Шымкент")]["amount"].sum()
rev_oct_shymkent = df_clean[mask_oct & (df_clean["city_clean"] == "Шымкент")]["amount"].sum()
upd_shymkent = (rev_oct_shymkent - rev_sep_shymkent) / rev_sep_shymkent
print("Шымкент:")
print(f"  Выручка за сентябрь: {rev_sep_shymkent:,.2f}")
print(f"  Выручка за октябрь: {rev_oct_shymkent:,.2f}")
print(f"  Изменение: {upd_shymkent:.2%}\n")

# Сравнение падения
ratio = upd_shymkent / upd_almaty
print(f"Падение в Шымкенте сильнее, чем в Алматы, в {ratio:.0f} раза.")

Алматы:
  Выручка за сентябрь: 15,523,940.00
  Выручка за октябрь: 14,643,020.00
  Изменение: -5.67%

Шымкент:
  Выручка за сентябрь: 4,949,800.00
  Выручка за октябрь: 4,044,010.00
  Изменение: -18.30%

Падение в Шымкенте сильнее, чем в Алматы, в 3 раза.


In [ ]:
## [Стандарт] 5. Исправьте код от AI
#AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
#cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
cancel_sh = df[(df["city_clean"] == "Шымкент") & (df["status"] == "отменён")]
print(len(cancel_sh))
print("Добавили недостающие скобки, тк сначала вычисляется по амперсанту \nВместо переменной city заменила на city_clean")

164
Добавили недостающие скобки, тк сначала вычисляется по амперсанту 
Вместо переменной city заменила на city_clean


In [ ]:
## [Стандарт] 6. Самые частые клиенты
# Кто сделал больше всего заказов (value_counts по customer_id)? Покажите все заказы этого клиента (loc с маской, столбцы дата, город, сумма, статус).
# Есть ли среди его заказов повторяющиеся order_id? Что это значит для выручки, которую мы считали весь урок?

customers = df[df["customer_id"] != "TEST"]["customer_id"].value_counts()
most_active_id = customers.index[0]
order_count = customers.iloc[0]
print(f"Самый частый клиент: {most_active_id} (Заказов: {order_count})\n")

# Показываем все заказы этого клиента
columns_to_show = ["order_date", "city_clean", "amount", "status", "order_id"]
client_orders = df.loc[df["customer_id"] == most_active_id, columns_to_show]
print("Все заказы клиента:")
print(client_orders)

# Проверяем на дубликаты order_id
print("Количество уникальных order_id:", client_orders["order_id"].nunique())
print("Общее количество строк у клиента:", len(client_orders))
print("\nЧто это значит для выручки:")
print("Наличие дубликатов по order_id означает, что в данных присутствуют повторы одинаковых заказов.")
print("Если мы просто суммировали столбец amount, мы искусственно завысили выручку, посчитав одни и те же заказы несколько раз.")

Самый частый клиент: C01818 (Заказов: 11)

Все заказы клиента:
            order_date city_clean   amount   status  order_id
57    2026-09-01 15:00    Шымкент   7350.0  оплачен    101125
779   2026-09-09 18:00    Шымкент  45270.0  оплачен    100096
1371  2026-09-15 21:00     Алматы   8890.0  оплачен    101504
1375  2026-09-15 21:00     Алматы   8890.0  оплачен    101504
2195  2026-09-24 18:00     Астана   1490.0  оплачен    101860
2617  2026-09-29 09:00     Астана  22170.0  оплачен    101462
2776  2026-09-30 21:00     Алматы  20180.0  оплачен    101495
3417  2026-10-08 14:00  Караганда   4330.0  оплачен    103416
4279  2026-10-18 08:00     Астана  13600.0  оплачен    102644
4610  2026-10-21 19:00     Алматы  21990.0  оплачен    104001
4778  2026-10-23 18:00     Алматы  11140.0  оплачен    104572
Количество уникальных order_id: 10
Общее количество строк у клиента: 11

Что это значит для выручки:
Наличие дубликатов по order_id означает, что в данных присутствуют повторы одинаковых заказо